# From Training to Production: Saving, Exporting, and Serving Models

You've trained models throughout this course - MLPs, CNNs, fine-tuned ResNets. But training is only half the story. A model that lives in a Jupyter notebook is useful for learning, not for production. At some point, you need to answer: **how do I actually use this model outside of this notebook?**

This lesson covers the full lifecycle of a trained model: what it actually IS (it's not one thing), how to save and load it, the different export formats and why they exist, and how to serve predictions. These concepts apply to any PyTorch model - image classifiers, YOLO detectors, tabular MLPs, recommenders, or language models.

**Prerequisites:** Any lesson from L5 onwards. We'll train a small model here so the notebook is self-contained.

In [1]:
# === Setup ===
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['font.size'] = 11

print("Setup loaded!")

Setup loaded!


Imports and a quick trained model to work with throughout the notebook.

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from pathlib import Path
from PIL import Image
import numpy as np
import time
import warnings
import json as _json

warnings.filterwarnings('ignore')
torch.manual_seed(42)

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"PyTorch {torch.__version__}, device: {device}")

PyTorch 2.9.1+cu128, device: cuda


We need a trained model to experiment with. Let's quickly train a small CNN on MNIST - nothing fancy, just enough to have real weights to save and export.

In [3]:
# Quick MNIST CNN - just to have a trained model
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128), nn.ReLU(),
            nn.Linear(128, 10)
        )
    
    def forward(self, x):
        return self.classifier(self.features(x))

# Train for 2 epochs - just enough to get reasonable weights
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_data = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST('./data', train=False, transform=transform)
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader = DataLoader(test_data, batch_size=128)

model = SmallCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(2):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = nn.CrossEntropyLoss()(model(images), labels)
        loss.backward()
        optimizer.step()
    
    model.eval()
    correct = sum((model(x.to(device)).argmax(1) == y.to(device)).sum().item() 
                  for x, y in test_loader)
    print(f"Epoch {epoch+1}: {correct/len(test_data)*100:.1f}% accuracy")

print(f"\nModel ready: {sum(p.numel() for p in model.parameters()):,} parameters")

Epoch 1: 97.7% accuracy


Epoch 2: 98.8% accuracy

Model ready: 206,922 parameters


## What IS a Trained Model?

This seems like a simple question, but the answer is surprisingly important. In fastai (L2), saving a model was one line: `learn.export('model.pkl')`. Loading was one line: `learn = load_learner('model.pkl')`. Everything just worked.

But what was actually in that pickle file? And why do things break when you try to load a model saved on a different machine, with a different Python version, or after refactoring your code?

A trained PyTorch model is **not one thing**. It's four separate pieces that must stay in sync:

In [4]:
# Let's look at what our trained model actually contains

# Piece 1: The architecture (Python code)
print("1. ARCHITECTURE (code)")
print(f"   Class: {model.__class__.__name__}")
print(f"   Layers: {sum(1 for _ in model.modules()) - 1}")
print()

# Piece 2: The trained weights (numbers)
print("2. TRAINED WEIGHTS (numbers)")
state = model.state_dict()
print(f"   state_dict has {len(state)} entries:")
for name, tensor in list(state.items())[:4]:
    print(f"     {name}: shape {list(tensor.shape)}, dtype {tensor.dtype}")
print(f"     ... ({len(state)} total)")
print()

# Piece 3: The preprocessing pipeline (code + config)
print("3. PREPROCESSING PIPELINE (code/config)")
print(f"   Transform: ToTensor -> Normalize(mean=0.1307, std=0.3081)")
print(f"   Input: 28x28 grayscale, values 0-1, then normalized")
print()

# Piece 4: The output mapping (data)
print("4. OUTPUT MAPPING (data)")
print(f"   Index 0 = digit '0', index 1 = digit '1', ..., index 9 = digit '9'")

1. ARCHITECTURE (code)
   Class: SmallCNN
   Layers: 12

2. TRAINED WEIGHTS (numbers)
   state_dict has 8 entries:
     features.0.weight: shape [16, 1, 3, 3], dtype torch.float32
     features.0.bias: shape [16], dtype torch.float32
     features.3.weight: shape [32, 16, 3, 3], dtype torch.float32
     features.3.bias: shape [32], dtype torch.float32
     ... (8 total)

3. PREPROCESSING PIPELINE (code/config)
   Transform: ToTensor -> Normalize(mean=0.1307, std=0.3081)
   Input: 28x28 grayscale, values 0-1, then normalized

4. OUTPUT MAPPING (data)
   Index 0 = digit '0', index 1 = digit '1', ..., index 9 = digit '9'


These four pieces are independent. The weights are just numbers - without the architecture to arrange them into layers, they're meaningless. The architecture is just code - without weights, it produces random outputs. The preprocessing defines how raw input becomes the tensor the model expects. The output mapping turns an index into a human-readable label.

**The core problem of deployment:** you must keep all four pieces synchronized. If any one is wrong or mismatched, you don't get an error - you get **silent wrong predictions**. The model happily produces outputs, they're just incorrect.

### The state_dict: What Are Weights, Really?

When we say "trained weights," what are we actually talking about? Let's look inside.

In [5]:
# The state_dict is an OrderedDict: parameter name -> tensor
state_dict = model.state_dict()

print(f"Type: {type(state_dict).__name__}")
print(f"Number of entries: {len(state_dict)}")
print()

# Each entry is a named tensor
for name, tensor in state_dict.items():
    print(f"{name:40s} shape={str(list(tensor.shape)):20s} values={tensor.numel():>6,} dtype={tensor.dtype}")

Type: OrderedDict
Number of entries: 8

features.0.weight                        shape=[16, 1, 3, 3]        values=   144 dtype=torch.float32
features.0.bias                          shape=[16]                 values=    16 dtype=torch.float32
features.3.weight                        shape=[32, 16, 3, 3]       values= 4,608 dtype=torch.float32
features.3.bias                          shape=[32]                 values=    32 dtype=torch.float32
classifier.1.weight                      shape=[128, 1568]          values=200,704 dtype=torch.float32
classifier.1.bias                        shape=[128]                values=   128 dtype=torch.float32
classifier.3.weight                      shape=[10, 128]            values= 1,280 dtype=torch.float32
classifier.3.bias                        shape=[10]                 values=    10 dtype=torch.float32


Every learnable parameter in the model has a name (derived from the Python attribute path) and a tensor of numbers. The convolutional weights are 4D tensors (out_channels x in_channels x height x width). The linear weights are 2D matrices. The biases are 1D vectors.

These are the numbers that training produced. The entire training process - forward passes, loss computation, backpropagation, optimizer steps across thousands of batches - all of it was just to find these specific numbers. Everything else (the architecture, the optimizer, the training loop) was scaffolding. The state_dict is the deliverable.

But there's a subtlety. The state_dict also contains **non-learnable parameters**:

In [6]:
# BatchNorm layers store running statistics that aren't learned via gradient descent
# They're computed during training as a running average

# Let's add a model with BatchNorm to show this
class CNNWithBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),  # <-- stores running_mean and running_var
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(16 * 14 * 14, 10))
    
    def forward(self, x):
        return self.classifier(self.features(x))

bn_model = CNNWithBN()
bn_state = bn_model.state_dict()

print("BatchNorm entries in state_dict:")
for name, tensor in bn_state.items():
    if 'bn' in name or 'batch' in name.lower():
        learned = "learned (gradient descent)" if 'weight' in name or 'bias' in name else "tracked (running average)"
        print(f"  {name:45s} {learned}")

print()
print("running_mean and running_var are NOT learned by backprop.")
print("They're computed as running averages during training.")
print("This is why model.eval() matters - it switches BN to use these stored stats.")

BatchNorm entries in state_dict:
  features.1.num_batches_tracked                tracked (running average)

running_mean and running_var are NOT learned by backprop.
They're computed as running averages during training.
This is why model.eval() matters - it switches BN to use these stored stats.


This is why `model.eval()` is critical before inference. In training mode, BatchNorm computes statistics from the current batch. In eval mode, it uses the stored running mean and variance from training. If you forget `model.eval()`, your predictions depend on which batch you're processing - they'll be noisy and inconsistent.

## Saving and Loading Models

Now that we understand what a model contains, how do we persist it?

### Strategy 1: Save the state_dict (recommended)

Save just the numbers. The architecture lives in your code.

In [7]:
# Save
torch.save(model.state_dict(), 'model_weights.pth')
print(f"Saved state_dict to model_weights.pth")
print(f"File size: {Path('model_weights.pth').stat().st_size / 1024:.1f} KB")

# Load
loaded_model = SmallCNN()  # must recreate the architecture first
loaded_model.load_state_dict(torch.load('model_weights.pth', weights_only=True))
loaded_model.eval()

print(f"\nLoaded successfully. Parameters match: {all(torch.equal(a.cpu(), b.cpu()) for a, b in zip(model.parameters(), loaded_model.parameters()))}")

Saved state_dict to model_weights.pth
File size: 812.2 KB

Loaded successfully. Parameters match: True


The `.pth` file contains only numbers - no Python code, no class definitions, no imports. This makes it:
- **Portable**: works across machines, OS versions, Python versions
- **Safe**: no arbitrary code execution when loading (with `weights_only=True`)
- **Small**: just the tensors, nothing else

The tradeoff: you must have the class definition (`SmallCNN`) available when loading. The file doesn't know what architecture it belongs to.

### What happens when architecture and weights don't match?

In [8]:
# What if we try to load weights into the wrong architecture?
class DifferentCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 32 channels, not 16
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(32 * 14 * 14, 10))
    
    def forward(self, x):
        return self.classifier(self.features(x))

wrong_model = DifferentCNN()
try:
    wrong_model.load_state_dict(torch.load('model_weights.pth', weights_only=True))
    print("Loaded successfully (this shouldn't happen)")
except RuntimeError as e:
    print("ERROR: Architecture mismatch!")
    print(f"  {str(e)[:200]}")
    print()
    print("This is a GOOD error - it tells you something is wrong.")
    print("Silent wrong predictions (from wrong preprocessing or class order) are much worse.")

ERROR: Architecture mismatch!
  Error(s) in loading state_dict for DifferentCNN:
	Unexpected key(s) in state_dict: "features.3.weight", "features.3.bias", "classifier.3.weight", "classifier.3.bias". 
	size mismatch for features.0.we

This is a GOOD error - it tells you something is wrong.
Silent wrong predictions (from wrong preprocessing or class order) are much worse.


### Strategy 2: Save the entire model (not recommended)

`torch.save(model)` pickles the entire Python object - architecture, weights, and all.

In [9]:
# Save entire model
torch.save(model, 'model_full.pkl')
print(f"Full model: {Path('model_full.pkl').stat().st_size / 1024:.1f} KB")

# Load - no need to know the class
loaded_full = torch.load('model_full.pkl', weights_only=False)
print(f"Loaded: {loaded_full.__class__.__name__}")

Full model: 815.2 KB
Loaded: SmallCNN


Seems easier, right? But this approach has serious problems:

**1. Pickle is fragile.** If you rename your class, move it to a different file, or change the module structure, the pickle breaks. It stores the full import path (`__main__.SmallCNN`), and if that path doesn't exist at load time, you get an error.

**2. Pickle is a security risk.** Loading a pickle file executes arbitrary Python code. A malicious `.pkl` file could delete your files or install malware. That's why PyTorch added `weights_only=True` for state_dict loading.

**3. Pickle is version-dependent.** A model pickled with Python 3.10 might not unpickle with Python 3.12. A model pickled with PyTorch 2.0 might not load with PyTorch 2.5.

**Rule of thumb:** always save `state_dict`, never `torch.save(model)` for anything you plan to keep or share.

### The Checkpoint Pattern

In practice, you want to save more than just weights. What if training crashes and you want to resume? What if you need to know which preprocessing was used? The checkpoint pattern bundles everything into one file:

In [10]:
# Save a complete checkpoint
checkpoint = {
    'model_state_dict': model.state_dict(),
    'model_class': 'SmallCNN',
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': 2,
    'accuracy': 0.98,
    
    # Preprocessing config - so you know exactly how to prepare inputs
    'input_size': 28,
    'channels': 1,
    'normalize_mean': (0.1307,),
    'normalize_std': (0.3081,),
    
    # Output mapping
    'class_names': [str(i) for i in range(10)],
}

torch.save(checkpoint, 'mnist_checkpoint.pth')
print(f"Checkpoint saved: {Path('mnist_checkpoint.pth').stat().st_size / 1024:.1f} KB")
print(f"Contains: {list(checkpoint.keys())}")

Checkpoint saved: 2435.7 KB
Contains: ['model_state_dict', 'model_class', 'optimizer_state_dict', 'epoch', 'accuracy', 'input_size', 'channels', 'normalize_mean', 'normalize_std', 'class_names']


Loading a checkpoint and using it:

In [11]:
# Load checkpoint
ckpt = torch.load('mnist_checkpoint.pth', weights_only=False)

print(f"Model class: {ckpt['model_class']}")
print(f"Trained for: {ckpt['epoch']} epochs")
print(f"Accuracy: {ckpt['accuracy']}")
print(f"Preprocessing: normalize(mean={ckpt['normalize_mean']}, std={ckpt['normalize_std']})")
print(f"Classes: {ckpt['class_names']}")

# Recreate and load
restored_model = SmallCNN()
restored_model.load_state_dict(ckpt['model_state_dict'])
restored_model.eval()
print(f"\nModel restored successfully")

Model class: SmallCNN
Trained for: 2 epochs
Accuracy: 0.98
Preprocessing: normalize(mean=(0.1307,), std=(0.3081,))
Classes: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']

Model restored successfully


The checkpoint solves the "4 pieces" problem: architecture info (class name), weights (state_dict), preprocessing config (normalization values), and output mapping (class names) are all in one file. The architecture itself is still code, but the checkpoint tells you which class to instantiate.

This is also how you resume training after a crash: load the checkpoint, restore both the model AND optimizer state, and continue from the saved epoch.

## Export Formats: Beyond PyTorch

A `.pth` file requires Python and PyTorch to load. That's fine for research and development. But what if you need to:

- Run inference in a C++ application?
- Deploy to a mobile phone or edge device?
- Serve predictions from a web browser?
- Use a different ML framework?

This is where export formats come in. They convert your model from "Python object" to "standalone computation graph" that can run without Python.

### ONNX: The Universal Format

**ONNX (Open Neural Network Exchange)** is an open standard for representing ML models. Think of it as a PDF for models - any tool that speaks ONNX can read it, regardless of which framework created it.

When you export to ONNX, PyTorch traces your model's forward pass and records every operation (convolution, ReLU, matrix multiply) as a graph. The result is a `.onnx` file that contains:
- The computation graph (which operations, in what order)
- The trained weights (embedded in the file)
- Input/output shapes and types

No Python code, no class definitions, no PyTorch dependency.

In [12]:
import onnx
import onnxruntime as ort

# Export to ONNX
model.eval()
model_cpu = model.cpu()

# Create a dummy input matching what the model expects
dummy_input = torch.randn(1, 1, 28, 28)

# Export - PyTorch traces the forward pass and records the computation graph
torch.onnx.export(
    model_cpu,
    dummy_input,
    'model.onnx',
    input_names=['image'],
    output_names=['prediction'],
    dynamic_axes={'image': {0: 'batch_size'}, 'prediction': {0: 'batch_size'}},  # allow variable batch
)

print(f"ONNX model saved: {Path('model.onnx').stat().st_size / 1024:.1f} KB")

# Verify the ONNX model is valid
onnx_model = onnx.load('model.onnx')
onnx.checker.check_model(onnx_model)
print("ONNX model verified: valid!")

W0408 00:33:40.015000 283503 /opt/miniforge3/miniforge3/envs/ml-venv/lib/python3.11/site-packages/torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.


W0408 00:33:40.016000 283503 /opt/miniforge3/miniforge3/envs/ml-venv/lib/python3.11/site-packages/torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.


W0408 00:33:40.016000 283503 /opt/miniforge3/miniforge3/envs/ml-venv/lib/python3.11/site-packages/torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


W0408 00:33:40.017000 283503 /opt/miniforge3/miniforge3/envs/ml-venv/lib/python3.11/site-packages/torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `SmallCNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SmallCNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 1 of general pattern rewrite rules.
ONNX model saved: 16.8 KB
ONNX model verified: valid!


The `dynamic_axes` parameter is important - it allows the batch dimension to vary. Without it, the model would only accept exactly one image at a time. With it, you can pass 1, 32, or 1000 images in a single call.

Now let's run inference with ONNX Runtime - a high-performance inference engine that works without PyTorch:

In [13]:
# Run inference with ONNX Runtime - no PyTorch needed
session = ort.InferenceSession('model.onnx')

# Prepare a test image (same preprocessing as training)
test_image = test_data[0][0].unsqueeze(0).numpy()  # (1, 1, 28, 28) as numpy

# Run inference
onnx_output = session.run(None, {'image': test_image})
onnx_pred = np.argmax(onnx_output[0])

# Compare to PyTorch
model_cpu.eval()
with torch.no_grad():
    pytorch_output = model_cpu(torch.tensor(test_image))
    pytorch_pred = pytorch_output.argmax(1).item()

print(f"PyTorch prediction: {pytorch_pred}")
print(f"ONNX prediction:    {onnx_pred}")
print(f"Match: {pytorch_pred == onnx_pred}")

PyTorch prediction: 7
ONNX prediction:    7
Match: True


Let's benchmark the speed difference:

In [14]:
# Benchmark: PyTorch vs ONNX Runtime
batch = torch.randn(64, 1, 28, 28)

# PyTorch
model_cpu.eval()
start = time.time()
for _ in range(100):
    with torch.no_grad():
        _ = model_cpu(batch)
pytorch_time = (time.time() - start) / 100

# ONNX Runtime
batch_np = batch.numpy()
start = time.time()
for _ in range(100):
    _ = session.run(None, {'image': batch_np})
onnx_time = (time.time() - start) / 100

print(f"PyTorch:      {pytorch_time*1000:.2f} ms per batch")
print(f"ONNX Runtime: {onnx_time*1000:.2f} ms per batch")
print(f"Speedup:      {pytorch_time/onnx_time:.1f}x")

PyTorch:      0.81 ms per batch
ONNX Runtime: 0.43 ms per batch
Speedup:      1.9x


ONNX Runtime is often faster than PyTorch for inference because it applies graph-level optimizations (fusing operations, eliminating redundancies) that PyTorch's eager execution doesn't. For production serving, this speedup matters.

### TorchScript: Staying in PyTorch-land

**TorchScript** is PyTorch's own serialization format. It JIT-compiles your model into an intermediate representation that can run without Python. Less universal than ONNX (only PyTorch ecosystem), but useful for mobile and C++ deployment.

In [15]:
# TorchScript via tracing
traced_model = torch.jit.trace(model_cpu, dummy_input)
traced_model.save('model_traced.pt')
print(f"TorchScript model: {Path('model_traced.pt').stat().st_size / 1024:.1f} KB")

# Load and run - no class definition needed
loaded_traced = torch.jit.load('model_traced.pt')
with torch.no_grad():
    ts_pred = loaded_traced(torch.tensor(test_image)).argmax(1).item()
print(f"TorchScript prediction: {ts_pred} (matches PyTorch: {ts_pred == pytorch_pred})")

TorchScript model: 832.6 KB


TorchScript prediction: 7 (matches PyTorch: True)


### When to Use Which Format

| Format | File | Needs Python? | Needs PyTorch? | Best for |
|---|---|---|---|---|
| **state_dict** | `.pth` | Yes | Yes | Development, research, checkpointing |
| **TorchScript** | `.pt` | No | Runtime only | Mobile, C++, PyTorch ecosystem |
| **ONNX** | `.onnx` | No | No | Production APIs, edge devices, cross-framework |

For most students in this course: `state_dict` for development, ONNX if you need production speed or non-Python deployment.

## The Inference Pipeline

Every deployed model follows the same steps, regardless of domain:

```
raw input -> preprocess -> tensor -> model(tensor) -> post-process -> response
```

The preprocessing must **exactly match** what was used during training. This is the most common deployment bug - and the hardest to debug because the model doesn't error, it just gives wrong answers.

Let's build a complete inference pipeline for our MNIST model:

In [16]:
def predict_digit(model, image_path_or_tensor, device='cpu'):
    """Complete inference pipeline: raw input to prediction."""
    model.eval()
    model = model.to(device)
    
    # 1. Preprocess - MUST match training
    if isinstance(image_path_or_tensor, (str, Path)):
        img = Image.open(image_path_or_tensor).convert('L')  # grayscale
        transform = transforms.Compose([
            transforms.Resize((28, 28)),
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,)),  # same as training!
        ])
        tensor = transform(img).unsqueeze(0)  # add batch dim
    else:
        tensor = image_path_or_tensor
    
    # 2. Inference
    tensor = tensor.to(device)
    with torch.no_grad():                  # no gradient tracking (saves memory)
        logits = model(tensor)             # raw model output
    
    # 3. Post-process
    probs = torch.softmax(logits, dim=1)   # convert to probabilities
    pred_class = probs.argmax(1).item()    # most likely class
    confidence = probs.max().item()         # how confident
    
    return pred_class, confidence

# Test it
digit, conf = predict_digit(model.cpu(), test_data[0][0].unsqueeze(0))
true_label = test_data[0][1]
print(f"Predicted: {digit} (confidence: {conf:.3f})")
print(f"Actual:    {true_label}")

Predicted: 7 (confidence: 1.000)
Actual:    7


### Common Inference Bugs

These are the mistakes that waste hours of debugging because the model doesn't crash - it just gives wrong results:

In [17]:
# Bug 1: Forgetting model.eval()
model_buggy = SmallCNN()
model_buggy.load_state_dict(torch.load('model_weights.pth', weights_only=True))
# model_buggy.eval()  <-- FORGOT THIS

# With dropout or batchnorm, train mode gives different results each time
model_buggy.train()  # explicitly in train mode
results_train = []
test_tensor = test_data[0][0].unsqueeze(0)
for _ in range(5):
    with torch.no_grad():
        out = model_buggy(test_tensor)
        results_train.append(out.softmax(1).max().item())

model_buggy.eval()  # correct mode
results_eval = []
for _ in range(5):
    with torch.no_grad():
        out = model_buggy(test_tensor)
        results_eval.append(out.softmax(1).max().item())

print("Bug: model.train() mode (inconsistent between runs):")
print(f"  Confidences: {[f'{r:.4f}' for r in results_train]}")
print(f"  All same? {len(set([f'{r:.4f}' for r in results_train])) == 1}")
print()
print("Fixed: model.eval() mode (consistent):")
print(f"  Confidences: {[f'{r:.4f}' for r in results_eval]}")
print(f"  All same? {len(set([f'{r:.4f}' for r in results_eval])) == 1}")

Bug: model.train() mode (inconsistent between runs):
  Confidences: ['1.0000', '1.0000', '1.0000', '1.0000', '1.0000']
  All same? True

Fixed: model.eval() mode (consistent):
  Confidences: ['1.0000', '1.0000', '1.0000', '1.0000', '1.0000']
  All same? True


Other common bugs to watch for:

In [18]:
# Bug 2: Forgetting torch.no_grad()
# Not wrong, but wastes memory tracking gradients you'll never use

# Bug 3: Wrong normalization
test_img = test_data[0][0].unsqueeze(0)

model.eval()
model_cpu = model.cpu()
with torch.no_grad():
    correct_pred = model_cpu(test_img).softmax(1)
    
    # What if we forget normalization entirely? (raw 0-1 pixels)
    raw_img = (test_img * 0.3081) + 0.1307  # undo normalization
    wrong_pred = model_cpu(raw_img).softmax(1)

print("Correct preprocessing:")
print(f"  Prediction: {correct_pred.argmax(1).item()}, confidence: {correct_pred.max().item():.3f}")
print()
print("Wrong preprocessing (forgot normalization):")
print(f"  Prediction: {wrong_pred.argmax(1).item()}, confidence: {wrong_pred.max().item():.3f}")
print()
print("No error, no warning - just wrong. This is why preprocessing must match training.")

Correct preprocessing:
  Prediction: 7, confidence: 1.000

Wrong preprocessing (forgot normalization):
  Prediction: 7, confidence: 0.895

No error, no warning - just wrong. This is why preprocessing must match training.


Other common bugs that cause silent failures:

In [19]:
# Bug 4: Loading GPU model on CPU (or vice versa)
# Save a model from GPU
model.to('cuda' if torch.cuda.is_available() else 'cpu')
torch.save(model.state_dict(), 'model_gpu.pth')

# Load on CPU - need map_location
loaded_cpu = SmallCNN()
loaded_cpu.load_state_dict(
    torch.load('model_gpu.pth', map_location='cpu', weights_only=True)
)
print("Loaded GPU weights onto CPU using map_location='cpu'")
print("Without map_location, this would error if CUDA isn't available")

Loaded GPU weights onto CPU using map_location='cpu'
Without map_location, this would error if CUDA isn't available


## Serving Models

You know FastAPI, Docker, nginx. The infrastructure is familiar from your devops background. The only new part is connecting the inference pipeline to an endpoint. Here's what the ML-specific code looks like:

In [20]:
# This is what a FastAPI model endpoint looks like (don't run, just read)
# You'd put this in a .py file, not a notebook

ENDPOINT_CODE = '''
from fastapi import FastAPI, UploadFile
import torch
from torchvision import transforms
from PIL import Image
import io

app = FastAPI()

# Load model ONCE at startup (not per request)
model = SmallCNN()
model.load_state_dict(torch.load("model_weights.pth", weights_only=True))
model.eval()

transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

@app.post("/predict")
async def predict(file: UploadFile):
    image = Image.open(io.BytesIO(await file.read())).convert("L")
    tensor = transform(image).unsqueeze(0)
    
    with torch.no_grad():
        probs = model(tensor).softmax(1)
    
    return {
        "prediction": int(probs.argmax(1)),
        "confidence": float(probs.max()),
        "probabilities": probs[0].tolist(),
    }
'''

print(ENDPOINT_CODE)
print("# That's ~25 lines of ML-specific code.")
print("# Everything else (routing, Docker, HTTPS) is standard web dev.")


from fastapi import FastAPI, UploadFile
import torch
from torchvision import transforms
from PIL import Image
import io

app = FastAPI()

# Load model ONCE at startup (not per request)
model = SmallCNN()
model.load_state_dict(torch.load("model_weights.pth", weights_only=True))
model.eval()

transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

@app.post("/predict")
async def predict(file: UploadFile):
    image = Image.open(io.BytesIO(await file.read())).convert("L")
    tensor = transform(image).unsqueeze(0)

    with torch.no_grad():
        probs = model(tensor).softmax(1)

    return {
        "prediction": int(probs.argmax(1)),
        "confidence": float(probs.max()),
        "probabilities": probs[0].tolist(),
    }

# That's ~25 lines of ML-specific code.
# Everything else (routing, Docker, HTTPS) is standard web dev.


Key decisions for serving:

**Load model at startup.** Model loading is slow (disk I/O + memory allocation). Do it once when the server starts, not per request.

**CPU vs GPU.** For low-throughput APIs (a few requests per second), CPU is fine and much cheaper. GPU matters when you need to process many images per second or run large models.

**Batching.** If your API gets bursts of requests, accumulating them into batches before running inference is much more efficient than processing one at a time. Frameworks like TorchServe and Triton handle this automatically.

**ONNX for production.** If latency matters, export to ONNX and serve with ONNX Runtime instead of PyTorch. Faster inference, smaller memory footprint, no Python GIL issues.

## MLOps: Keeping Models Alive

Deploying a model is not the end. Models degrade over time because the world changes. This section is vocabulary and awareness - you won't implement these systems in this course, but you'll encounter them in industry.

**Model versioning.** Which weights go with which code? The checkpoint pattern helps (metadata in the file), but at scale you need tracking tools. MLflow and Weights & Biases are the common ones - they log every training run with its hyperparameters, metrics, and artifacts.

**Reproducibility.** "It works on my machine" is worse in ML than in web dev. A model's behavior depends on random seeds, package versions, data order, and hardware. Pinning dependencies, saving hyperparameters, and using deterministic settings helps but doesn't fully solve it.

**Data drift.** Your model was trained on data from January. By June, the real-world data looks different - new patterns, new edge cases, shifted distributions. The model's accuracy quietly drops. This is called **input drift** and it's the #1 reason deployed models fail silently.

**Monitoring.** How do you know the model is still working? You can't check accuracy in production (you don't have labels). Instead, you monitor proxy signals: prediction confidence distributions, input feature distributions, prediction class balance. If these shift, something changed.

**Retraining.** The feedback loop: collect production data, label a sample, retrain, compare to the current model, deploy if better. How often? Depends on how fast your domain changes. Some teams retrain weekly, some monthly, some only when monitoring triggers an alert.

## Summary

| Concept | Key Point |
|---|---|
| **A trained model** | 4 pieces: architecture (code), weights (numbers), preprocessing (config), output mapping (data) |
| **state_dict** | OrderedDict of parameter tensors. Save this, not the whole model. |
| **Checkpoint** | state_dict + metadata (preprocessing, class names, epoch, accuracy) |
| **pickle (torch.save(model))** | Fragile, insecure, version-dependent. Avoid for anything permanent. |
| **ONNX** | Framework-agnostic export. Faster inference, no Python needed. |
| **TorchScript** | PyTorch-native export. No Python needed, but PyTorch runtime required. |
| **model.eval()** | Switches BatchNorm and Dropout to inference mode. Always call before predicting. |
| **Preprocessing** | Must exactly match training. Silent wrong predictions if mismatched. |
| **Data drift** | Models degrade over time as the real world changes. Monitor and retrain. |

## Questionnaire

1. What are the four pieces of a trained model? Why must they stay in sync?
2. What is a state_dict and what does it contain beyond learnable weights?
3. Why is `torch.save(model.state_dict())` preferred over `torch.save(model)`?
4. What's the checkpoint pattern and when would you use it?
5. What's the difference between ONNX and TorchScript? When would you choose each?
6. Why does `model.eval()` matter? What goes wrong if you forget it?
7. What's the most common deployment bug and why is it hard to catch?
8. What is data drift and why do deployed models degrade over time?
9. Why load the model at server startup instead of per request?
10. What does `weights_only=True` do and why is it a security measure?

## Further Research

- **ONNX optimization** - ONNX Runtime supports graph optimizations, quantization (INT8), and hardware-specific backends (TensorRT for NVIDIA, DirectML for Windows)
- **Model compression** - pruning (removing unnecessary weights), quantization (reducing precision from float32 to int8), and distillation (training a smaller model to mimic a larger one)
- **MLflow** - open-source platform for tracking experiments, packaging models, and serving them
- **TorchServe** - PyTorch's official model serving framework with batching, versioning, and monitoring
- **Triton Inference Server** - NVIDIA's production serving solution for multi-model, multi-framework deployment

<div style="text-align: center; color: #888; font-size: 0.85em; margin-top: 40px; padding-top: 10px; border-top: 1px solid #ddd;">
&copy; 2025 Utvecklarakademin UA Aktiebolag. All rights reserved.<br>
This material is proprietary and may not be reproduced, distributed, or shared without written permission.
</div>